# Phase 3 — Bronze Ingestion & Validation

## Objective

This notebook validates the raw equipment operations data stored in the
Bronze layer of the Microsoft Fabric Lakehouse.

The Bronze layer preserves source data with minimal transformation.

### Activities
- Read all raw CSV files from `Files/bronze`
- Validate file accessibility
- Inspect schemas
- Validate row counts
- Check for empty datasets
- Prepare Bronze Delta tables for downstream Silver processing

### Source Datasets
- sites
- assets
- sensor_readings
- failures
- maintenance
- work_orders
- costs

### Step 1 - Read the data

In [2]:
from pyspark.sql import functions as F

bronze_path = "Files/bronze"
## read all seven files
files = {
    "sites": f"{bronze_path}/sites.csv",
    "assets": f"{bronze_path}/assets.csv",
    "sensor_readings": f"{bronze_path}/sensor_readings.csv",
    "failures": f"{bronze_path}/failures.csv",
    "maintenance": f"{bronze_path}/maintenance.csv",
    "work_orders": f"{bronze_path}/work_orders.csv",
    "costs": f"{bronze_path}/costs.csv"
}

print("Bronze file paths configured successfully.")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 4, Finished, Available, Finished, False)

Bronze file paths configured successfully.


In [3]:
bronze_dfs = {}

for name, path in files.items():
    
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(path)
    )
    
    bronze_dfs[name] = df
    
    print(
        f"{name:<20} "
        f"Rows: {df.count():>10} | "
        f"Columns: {len(df.columns)}"
    )

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 5, Finished, Available, Finished, False)

sites                Rows:          3 | Columns: 7
assets               Rows:        150 | Columns: 14
sensor_readings      Rows:     324000 | Columns: 12
failures             Rows:         40 | Columns: 11
maintenance          Rows:        310 | Columns: 13
work_orders          Rows:        310 | Columns: 16
costs                Rows:        310 | Columns: 14


In [4]:
sites_df = bronze_dfs["sites"]
assets_df = bronze_dfs["assets"]
sensor_readings_df = bronze_dfs["sensor_readings"]
failures_df = bronze_dfs["failures"]
maintenance_df = bronze_dfs["maintenance"]
work_orders_df = bronze_dfs["work_orders"]
costs_df = bronze_dfs["costs"]

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 6, Finished, Available, Finished, False)

In [5]:
display(sites_df.limit(10))


StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8309dd17-257b-400e-a9a5-4c8ad79a50a3)

### Step 2 — Bronze schema and basic quality validation

In [6]:
for name, df in bronze_dfs.items():
    print(f"\n{'='*60}")
    print(f"DATASET: {name.upper()}")
    print(f"{'='*60}")
    
    print(f"Rows    : {df.count()}")
    print(f"Columns : {len(df.columns)}")
    
    print("\nSchema:")
    df.printSchema()

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 8, Finished, Available, Finished, False)


DATASET: SITES
Rows    : 3
Columns : 7

Schema:
root
 |-- Site_ID: string (nullable = true)
 |-- Site_Name: string (nullable = true)
 |-- Location: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Site_Type: string (nullable = true)
 |-- Operational_Status: string (nullable = true)
 |-- Commission_Date: date (nullable = true)


DATASET: ASSETS
Rows    : 150
Columns : 14

Schema:
root
 |-- Asset_ID: string (nullable = true)
 |-- Site_ID: string (nullable = true)
 |-- Asset_Name: string (nullable = true)
 |-- Asset_Type: string (nullable = true)
 |-- Manufacturer: string (nullable = true)
 |-- Model: string (nullable = true)
 |-- Serial_Number: string (nullable = true)
 |-- Install_Date: date (nullable = true)
 |-- Criticality: string (nullable = true)
 |-- Rated_Capacity: double (nullable = true)
 |-- Capacity_Unit: string (nullable = true)
 |-- Asset_Status: string (nullable = true)
 |-- Expected_Life_Years: integer (nullable = true)
 |-- Warranty_End_Date: date (nu

In [7]:
from pyspark.sql import functions as F
## null-count check
for name, df in bronze_dfs.items():
    print(f"\n{'='*60}")
    print(f"NULL CHECK: {name.upper()}")
    print(f"{'='*60}")
    
    null_counts = df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    
    display(null_counts)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 9, Finished, Available, Finished, False)


NULL CHECK: SITES


SynapseWidget(Synapse.DataFrame, e1b49d42-c1e0-40df-908e-4fc468b3d967)


NULL CHECK: ASSETS


SynapseWidget(Synapse.DataFrame, 8936a748-c4ef-4000-af9e-3985861605fb)

...

SynapseWidget(Synapse.DataFrame, de0ca967-4a33-4609-b4ea-b7bd5c7b1bdb)


NULL CHECK: WORK_ORDERS


SynapseWidget(Synapse.DataFrame, 901dd2e1-48d0-4ee4-b7b9-81e8e8f1c46e)


NULL CHECK: COSTS


SynapseWidget(Synapse.DataFrame, ed65218d-bc97-4fc0-bac9-a047ae1b4089)

In [8]:
## check duplicates
for name, df in bronze_dfs.items():
    total_rows = df.count()
    distinct_rows = df.distinct().count()
    duplicate_rows = total_rows - distinct_rows
    
    print(
        f"{name:<20} "
        f"Total: {total_rows:<8} "
        f"Distinct: {distinct_rows:<8} "
        f"Duplicates: {duplicate_rows}"
    )

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 10, Finished, Available, Finished, False)

sites                Total: 3        Distinct: 3        Duplicates: 0
assets               Total: 150      Distinct: 150      Duplicates: 0
sensor_readings      Total: 324000   Distinct: 324000   Duplicates: 0
failures             Total: 40       Distinct: 40       Duplicates: 0
maintenance          Total: 310      Distinct: 310      Duplicates: 0
work_orders          Total: 310      Distinct: 310      Duplicates: 0
costs                Total: 310      Distinct: 310      Duplicates: 0


### Step 3 — Write Bronze Delta Tables.

In [9]:
for name, df in bronze_dfs.items():
    
    table_name = f"bronze_{name}"
    
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )
    
    print(f"Created table: {table_name}")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 11, Finished, Available, Finished, False)

Created table: bronze_sites
Created table: bronze_assets
Created table: bronze_sensor_readings
Created table: bronze_failures
Created table: bronze_maintenance
Created table: bronze_work_orders
Created table: bronze_costs


In [10]:
bronze_tables = [
    "bronze_sites",
    "bronze_assets",
    "bronze_sensor_readings",
    "bronze_failures",
    "bronze_maintenance",
    "bronze_work_orders",
    "bronze_costs"
]

for table in bronze_tables:
    count = spark.table(table).count()
    print(f"{table:<30} Rows: {count}")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 12, Finished, Available, Finished, False)

bronze_sites                   Rows: 3
bronze_assets                  Rows: 150
bronze_sensor_readings         Rows: 324000
bronze_failures                Rows: 40
bronze_maintenance             Rows: 310
bronze_work_orders             Rows: 310
bronze_costs                   Rows: 310


### Step 4 — Data Quality Rules & Quarantine.

In [13]:
from pyspark.sql import functions as F

work_orders_bronze = spark.table("bronze_work_orders")

invalid_work_orders = work_orders_bronze.filter(
    F.col("Scheduled_DateTime") < F.col("Created_DateTime")
)

valid_work_orders = work_orders_bronze.filter(
    (F.col("Scheduled_DateTime") >= F.col("Created_DateTime")) |
    F.col("Scheduled_DateTime").isNull()
)

print("Invalid work orders:", invalid_work_orders.count())
print("Valid work orders:", valid_work_orders.count())

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 15, Finished, Available, Finished, False)

Invalid work orders: 0
Valid work orders: 310


In [14]:
display(
    invalid_work_orders.select(
        "WorkOrder_ID",
        "Asset_ID",
        "Created_DateTime",
        "Scheduled_DateTime"
    )
)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 16, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, cb1fe7d8-acbb-435b-91da-5f3959d4b586)

In [15]:
quarantine_work_orders = (
    invalid_work_orders
    .withColumn(
        "DQ_Rule",
        F.lit("WO_SCHEDULED_BEFORE_CREATED")
    )
    .withColumn(
        "DQ_Reason",
        F.lit("Scheduled_DateTime is earlier than Created_DateTime")
    )
    .withColumn(
        "Quarantine_Timestamp",
        F.current_timestamp()
    )
)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 17, Finished, Available, Finished, False)

In [16]:
(
    quarantine_work_orders.write
    .format("delta")
    .mode("overwrite")
    .save("Files/quarantine/work_orders")
)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 18, Finished, Available, Finished, False)

In [17]:
quarantine_check = (
    spark.read
    .format("delta")
    .load("Files/quarantine/work_orders")
)

print("Quarantined rows:", quarantine_check.count())

display(quarantine_check)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 19, Finished, Available, Finished, False)

Quarantined rows: 0


SynapseWidget(Synapse.DataFrame, 48303933-5e72-4573-9b0a-ba0c691bdb54)

In [18]:
from pyspark.sql import functions as F

wo_check = (
    work_orders_bronze
    .withColumn(
        "Created_TS",
        F.to_timestamp("Created_DateTime")
    )
    .withColumn(
        "Scheduled_TS",
        F.to_timestamp("Scheduled_DateTime")
    )
)

print(
    "Null Created timestamps:",
    wo_check.filter(F.col("Created_TS").isNull()).count()
)

print(
    "Null Scheduled timestamps:",
    wo_check.filter(F.col("Scheduled_TS").isNull()).count()
)

print(
    "Scheduled before Created:",
    wo_check.filter(
        F.col("Scheduled_TS") < F.col("Created_TS")
    ).count()
)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 20, Finished, Available, Finished, False)

Null Created timestamps: 0
Null Scheduled timestamps: 0
Scheduled before Created: 0


In [19]:
display(
    wo_check
    .withColumn(
        "Schedule_Delay_Hours",
        (
            F.col("Scheduled_TS").cast("long") -
            F.col("Created_TS").cast("long")
        ) / 3600
    )
    .select(
        "WorkOrder_ID",
        "Created_DateTime",
        "Scheduled_DateTime",
        "Schedule_Delay_Hours"
    )
    .orderBy("Schedule_Delay_Hours")
    .limit(10)
)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 711b5ff5-9a41-49a6-ba64-52c6e4b8b3a8)

### Step 5 - Full Bronze Data Quality Validation

In [20]:
from pyspark.sql import functions as F

print("=== FOREIGN KEY CHECKS ===")

# Assets → Sites
invalid_asset_sites = (
    assets_df
    .join(
        sites_df.select("Site_ID"),
        on="Site_ID",
        how="left_anti"
    )
    .count()
)

# Sensor Readings → Assets
invalid_sensor_assets = (
    sensor_readings_df
    .join(
        assets_df.select("Asset_ID"),
        on="Asset_ID",
        how="left_anti"
    )
    .count()
)

# Failures → Assets
invalid_failure_assets = (
    failures_df
    .join(
        assets_df.select("Asset_ID"),
        on="Asset_ID",
        how="left_anti"
    )
    .count()
)

# Maintenance → Assets
invalid_maintenance_assets = (
    maintenance_df
    .join(
        assets_df.select("Asset_ID"),
        on="Asset_ID",
        how="left_anti"
    )
    .count()
)

# Work Orders → Assets
invalid_workorder_assets = (
    work_orders_df
    .join(
        assets_df.select("Asset_ID"),
        on="Asset_ID",
        how="left_anti"
    )
    .count()
)

# Costs → Assets
invalid_cost_assets = (
    costs_df
    .join(
        assets_df.select("Asset_ID"),
        on="Asset_ID",
        how="left_anti"
    )
    .count()
)

print("assets.Site_ID            :", invalid_asset_sites)
print("sensor_readings.Asset_ID :", invalid_sensor_assets)
print("failures.Asset_ID        :", invalid_failure_assets)
print("maintenance.Asset_ID     :", invalid_maintenance_assets)
print("work_orders.Asset_ID     :", invalid_workorder_assets)
print("costs.Asset_ID           :", invalid_cost_assets)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 22, Finished, Available, Finished, False)

=== FOREIGN KEY CHECKS ===
assets.Site_ID            : 0
sensor_readings.Asset_ID : 0
failures.Asset_ID        : 0
maintenance.Asset_ID     : 0
work_orders.Asset_ID     : 0
costs.Asset_ID           : 0


In [22]:
print("\n=== BUSINESS RULE CHECKS ===")

# Asset criticality
invalid_criticality = assets_df.filter(
    ~F.col("Criticality").isin(
        "Low",
        "Medium",
        "High",
        "Critical"
    )
).count()

# Sensor load %
invalid_load = sensor_readings_df.filter(
    (F.col("Load_Pct") < 0) |
    (F.col("Load_Pct") > 100)
).count()

# Health score
invalid_health = sensor_readings_df.filter(
    (F.col("Health_Score") < 0) |
    (F.col("Health_Score") > 100)
).count()

print("Invalid Criticality :", invalid_criticality)
print("Invalid Load Pct    :", invalid_load)
print("Invalid Health Score:", invalid_health)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 24, Finished, Available, Finished, False)


=== BUSINESS RULE CHECKS ===
Invalid Criticality : 0
Invalid Load Pct    : 0
Invalid Health Score: 0


### Step 6 — Create Silver Tables

In [23]:
silver_sites = spark.table("bronze_sites")
silver_assets = spark.table("bronze_assets")
silver_sensor_readings = spark.table("bronze_sensor_readings")
silver_failures = spark.table("bronze_failures")
silver_maintenance = spark.table("bronze_maintenance")
silver_work_orders = spark.table("bronze_work_orders")
silver_costs = spark.table("bronze_costs")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 25, Finished, Available, Finished, False)

In [24]:
silver_dfs = {
    "silver_sites": silver_sites,
    "silver_assets": silver_assets,
    "silver_sensor_readings": silver_sensor_readings,
    "silver_failures": silver_failures,
    "silver_maintenance": silver_maintenance,
    "silver_work_orders": silver_work_orders,
    "silver_costs": silver_costs
}

for table_name, df in silver_dfs.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"Created: {table_name}")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 26, Finished, Available, Finished, False)

Created: silver_sites
Created: silver_assets
Created: silver_sensor_readings
Created: silver_failures
Created: silver_maintenance
Created: silver_work_orders
Created: silver_costs


In [25]:
for table_name in silver_dfs.keys():
    count = spark.table(table_name).count()
    print(f"{table_name:<30} Rows: {count}")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 27, Finished, Available, Finished, False)

silver_sites                   Rows: 3
silver_assets                  Rows: 150
silver_sensor_readings         Rows: 324000
silver_failures                Rows: 40
silver_maintenance             Rows: 310
silver_work_orders             Rows: 310
silver_costs                   Rows: 310


### Step 7 — Build the Gold dimensional model

In [27]:
gold_dim_site = (
    spark.table("silver_sites")
    .select(
        "Site_ID",
        "Site_Name",
        "Location",
        "Region",
        "Site_Type",
        "Operational_Status",
        "Commission_Date"
    )
    .dropDuplicates(["Site_ID"])
)

gold_dim_asset = (
    spark.table("silver_assets")
    .select(
        "Asset_ID",
        "Site_ID",
        "Asset_Name",
        "Asset_Type",
        "Manufacturer",
        "Model",
        "Criticality",
        "Asset_Status"
    )
    .dropDuplicates(["Asset_ID"])
)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 29, Finished, Available, Finished, False)

In [28]:
print(spark.table("silver_assets").columns)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 30, Finished, Available, Finished, False)

['Asset_ID', 'Site_ID', 'Asset_Name', 'Asset_Type', 'Manufacturer', 'Model', 'Serial_Number', 'Install_Date', 'Criticality', 'Rated_Capacity', 'Capacity_Unit', 'Asset_Status', 'Expected_Life_Years', 'Warranty_End_Date']


In [29]:
(
    gold_dim_site.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_dim_site")
)

(
    gold_dim_asset.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_dim_asset")
)

print("Gold dimensions created.")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 31, Finished, Available, Finished, False)

Gold dimensions created.


In [30]:
gold_fact_sensor_reading = spark.table("silver_sensor_readings")
gold_fact_failure = spark.table("silver_failures")
gold_fact_maintenance = spark.table("silver_maintenance")
gold_fact_work_order = spark.table("silver_work_orders")
gold_fact_cost = spark.table("silver_costs")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 32, Finished, Available, Finished, False)

In [31]:
gold_facts = {
    "gold_fact_sensor_reading": gold_fact_sensor_reading,
    "gold_fact_failure": gold_fact_failure,
    "gold_fact_maintenance": gold_fact_maintenance,
    "gold_fact_work_order": gold_fact_work_order,
    "gold_fact_cost": gold_fact_cost
}

for table_name, df in gold_facts.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(table_name)
    )

    print(f"Created: {table_name}")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 33, Finished, Available, Finished, False)

Created: gold_fact_sensor_reading
Created: gold_fact_failure
Created: gold_fact_maintenance
Created: gold_fact_work_order
Created: gold_fact_cost


In [32]:
gold_tables = [
    "gold_dim_site",
    "gold_dim_asset",
    "gold_fact_sensor_reading",
    "gold_fact_failure",
    "gold_fact_maintenance",
    "gold_fact_work_order",
    "gold_fact_cost"
]

for table in gold_tables:
    print(f"{table:<30} Rows: {spark.table(table).count()}")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 34, Finished, Available, Finished, False)

gold_dim_site                  Rows: 3
gold_dim_asset                 Rows: 150
gold_fact_sensor_reading       Rows: 324000
gold_fact_failure              Rows: 40
gold_fact_maintenance          Rows: 310
gold_fact_work_order           Rows: 310
gold_fact_cost                 Rows: 310


### Step 8 — Gold relationship validation

In [33]:
print("=== DIMENSION KEY CHECKS ===")

dim_site = spark.table("gold_dim_site")
dim_asset = spark.table("gold_dim_asset")

site_total = dim_site.count()
site_distinct = dim_site.select("Site_ID").distinct().count()

asset_total = dim_asset.count()
asset_distinct = dim_asset.select("Asset_ID").distinct().count()

print("gold_dim_site")
print("Total rows     :", site_total)
print("Distinct Site_ID:", site_distinct)
print("Duplicate keys :", site_total - site_distinct)

print("\ngold_dim_asset")
print("Total rows      :", asset_total)
print("Distinct Asset_ID:", asset_distinct)
print("Duplicate keys  :", asset_total - asset_distinct)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 35, Finished, Available, Finished, False)

=== DIMENSION KEY CHECKS ===
gold_dim_site
Total rows     : 3
Distinct Site_ID: 3
Duplicate keys : 0

gold_dim_asset
Total rows      : 150
Distinct Asset_ID: 150
Duplicate keys  : 0


In [34]:
invalid_asset_sites = (
    dim_asset
    .join(
        dim_site.select("Site_ID"),
        on="Site_ID",
        how="left_anti"
    )
)

print(
    "Assets with invalid Site_ID:",
    invalid_asset_sites.count()
)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 36, Finished, Available, Finished, False)

Assets with invalid Site_ID: 0


In [35]:
fact_tables = {
    "gold_fact_sensor_reading": "Asset_ID",
    "gold_fact_failure": "Asset_ID",
    "gold_fact_maintenance": "Asset_ID",
    "gold_fact_work_order": "Asset_ID",
    "gold_fact_cost": "Asset_ID"
}

print("=== FACT → ASSET RELATIONSHIP CHECKS ===")

for table_name, key_col in fact_tables.items():

    fact_df = spark.table(table_name)

    invalid_count = (
        fact_df
        .join(
            dim_asset.select("Asset_ID"),
            fact_df[key_col] == dim_asset["Asset_ID"],
            "left_anti"
        )
        .count()
    )

    print(
        f"{table_name:<30} "
        f"Invalid Asset_ID: {invalid_count}"
    )

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 37, Finished, Available, Finished, False)

=== FACT → ASSET RELATIONSHIP CHECKS ===
gold_fact_sensor_reading       Invalid Asset_ID: 0
gold_fact_failure              Invalid Asset_ID: 0
gold_fact_maintenance          Invalid Asset_ID: 0
gold_fact_work_order           Invalid Asset_ID: 0
gold_fact_cost                 Invalid Asset_ID: 0


In [36]:
print("=== GOLD ROW COUNT CHECK ===")

for table in [
    "gold_dim_site",
    "gold_dim_asset",
    "gold_fact_sensor_reading",
    "gold_fact_failure",
    "gold_fact_maintenance",
    "gold_fact_work_order",
    "gold_fact_cost"
]:
    print(
        f"{table:<30}",
        spark.table(table).count()
    )

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 38, Finished, Available, Finished, False)

=== GOLD ROW COUNT CHECK ===
gold_dim_site                  3
gold_dim_asset                 150
gold_fact_sensor_reading       324000
gold_fact_failure              40
gold_fact_maintenance          310
gold_fact_work_order           310
gold_fact_cost                 310


### Step 9 — KPI-Ready Gold Validation.

In [37]:
for table in [
    "gold_fact_failure",
    "gold_fact_maintenance",
    "gold_fact_work_order",
    "gold_fact_cost",
    "gold_fact_sensor_reading"
]:
    print(f"\n{table.upper()}")
    print(spark.table(table).columns)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 39, Finished, Available, Finished, False)


GOLD_FACT_FAILURE
['Failure_ID', 'Asset_ID', 'Failure_DateTime', 'Failure_Type', 'Failure_Component', 'Severity', 'Downtime_Hours', 'Detection_Method', 'Root_Cause', 'Production_Impact', 'Failure_Status']

GOLD_FACT_MAINTENANCE
['Maintenance_ID', 'Asset_ID', 'Failure_ID', 'Maintenance_Date', 'Maintenance_Type', 'Maintenance_Reason', 'Component', 'Technician_ID', 'Duration_Hours', 'Planned_Flag', 'Maintenance_Status', 'Parts_Replaced', 'Notes']

GOLD_FACT_WORK_ORDER
['WorkOrder_ID', 'Asset_ID', 'Maintenance_ID', 'Failure_ID', 'Created_DateTime', 'Scheduled_DateTime', 'Completed_DateTime', 'WorkOrder_Type', 'Priority', 'Assigned_Team', 'WorkOrder_Status', 'Estimated_Hours', 'Actual_Hours', 'Description', 'SLA_Target_Hours', 'SLA_Breached_Flag']

GOLD_FACT_COST
['Cost_ID', 'Asset_ID', 'WorkOrder_ID', 'Maintenance_ID', 'Failure_ID', 'Cost_Date', 'Cost_Type', 'Labour_Cost', 'Parts_Cost', 'Contractor_Cost', 'Downtime_Cost', 'Other_Cost', 'Total_Cost', 'Currency']

GOLD_FACT_SENSOR_READING
[

In [38]:
from pyspark.sql import functions as F

failure_df = spark.table("gold_fact_failure")
maintenance_df = spark.table("gold_fact_maintenance")
work_order_df = spark.table("gold_fact_work_order")
cost_df = spark.table("gold_fact_cost")
sensor_df = spark.table("gold_fact_sensor_reading")

print("=== KPI READY GOLD VALIDATION ===")

# Failure KPIs
failure_kpis = failure_df.agg(
    F.count("*").alias("Failure_Count"),
    F.sum("Downtime_Hours").alias("Total_Downtime_Hours"),
    F.avg("Downtime_Hours").alias("Avg_Downtime_Hours")
)

# Maintenance KPIs
maintenance_kpis = maintenance_df.agg(
    F.count("*").alias("Maintenance_Count"),
    F.sum("Duration_Hours").alias("Total_Maintenance_Hours"),
    F.avg("Duration_Hours").alias("Avg_Maintenance_Hours")
)

# Work Order KPIs
work_order_kpis = work_order_df.agg(
    F.count("*").alias("Work_Order_Count"),
    F.avg("Actual_Hours").alias("Avg_Actual_Hours"),
    F.sum(
        F.when(F.col("SLA_Breached_Flag") == 1, 1).otherwise(0)
    ).alias("SLA_Breached_Count")
)

# Cost KPIs
cost_kpis = cost_df.agg(
    F.sum("Total_Cost").alias("Total_Cost"),
    F.avg("Total_Cost").alias("Avg_Cost_Per_Record"),
    F.sum("Downtime_Cost").alias("Total_Downtime_Cost")
)

# Sensor KPIs
sensor_kpis = sensor_df.agg(
    F.avg("Health_Score").alias("Avg_Health_Score"),
    F.avg("Load_Pct").alias("Avg_Load_Pct"),
    F.sum(
        F.when(F.col("Anomaly_Flag") == 1, 1).otherwise(0)
    ).alias("Anomaly_Count")
)

display(failure_kpis)
display(maintenance_kpis)
display(work_order_kpis)
display(cost_kpis)
display(sensor_kpis)

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 40, Finished, Available, Finished, False)

=== KPI READY GOLD VALIDATION ===


SynapseWidget(Synapse.DataFrame, 3e1370fd-209c-4e38-b54f-d9ebb6eebbf9)

SynapseWidget(Synapse.DataFrame, 9f488d28-6840-41cf-aa7c-0042fe6b7669)

SynapseWidget(Synapse.DataFrame, 12ff65af-9157-4044-b008-49f21dff5833)

SynapseWidget(Synapse.DataFrame, 07fff3f9-d50f-4c2a-b0f7-dc1a6cac4865)

SynapseWidget(Synapse.DataFrame, 4ed43f6f-d620-4060-8ca9-85067d645c0c)

In [39]:
print("=== GOLD KPI COLUMN NULL CHECK ===")

checks = {
    "failure.Downtime_Hours":
        failure_df.filter(F.col("Downtime_Hours").isNull()).count(),

    "maintenance.Duration_Hours":
        maintenance_df.filter(F.col("Duration_Hours").isNull()).count(),

    "work_order.Actual_Hours":
        work_order_df.filter(F.col("Actual_Hours").isNull()).count(),

    "cost.Total_Cost":
        cost_df.filter(F.col("Total_Cost").isNull()).count(),

    "sensor.Health_Score":
        sensor_df.filter(F.col("Health_Score").isNull()).count()
}

for name, count in checks.items():
    print(f"{name:<35}: {count}")

StatementMeta(, b0dc6ea3-120e-45cc-87c8-505cca1bde7c, 41, Finished, Available, Finished, False)

=== GOLD KPI COLUMN NULL CHECK ===
failure.Downtime_Hours             : 0
maintenance.Duration_Hours         : 0
work_order.Actual_Hours            : 0
cost.Total_Cost                    : 0
sensor.Health_Score                : 0


### Step 10 — Fabric Warehouse Serving Layer

* Created named 'EquipmentOperations_Warehouse'

* For the Warehouse mart, we will eventually serve only the business-focused tables:
dim_site
dim_asset

fact_failure
fact_maintenance
fact_work_order
fact_cost


### Step 11 — Load Gold business tables into the Fabric Warehouse